In [1]:
## 1. Import Libraries and Set Reproducibility

import os
import json
import random
import zipfile

import numpy as np
import pandas as pd
import tensorflow as tf

# Fixed random seed
RANDOM_SEED = 42

random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
tf.keras.utils.set_random_seed(RANDOM_SEED)

# Request deterministic TensorFlow operations
try:
    tf.config.experimental.enable_op_determinism()
except Exception:
    pass

print("Random seed set to:", RANDOM_SEED)

Random seed set to: 42


In [2]:
## 2. Load Official Data and EDA Records

# Local project folders
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))

zip_path = os.path.join(PROJECT_ROOT,"data","TBX11K.zip")

eda_output_dir = os.path.join(PROJECT_ROOT, "outputs", "eda")

preprocessing_output_dir = os.path.join( PROJECT_ROOT, "outputs", "preprocessing")

os.makedirs( preprocessing_output_dir, exist_ok=True)

# Load official training and validation lists
with zipfile.ZipFile(zip_path, "r") as zip_file:

    train_files = zip_file.read( "TBX11K/lists/TBX11K_train.txt").decode("utf-8").splitlines()

    val_files = zip_file.read( "TBX11K/lists/TBX11K_val.txt").decode("utf-8").splitlines()

train_files = [ path.strip() for path in train_files if path.strip()]

val_files = [ path.strip() for path in val_files if path.strip()]

# Load EDA records
eda_df = pd.read_csv( os.path.join( eda_output_dir, "eda_image_statistics.csv" ))

cross_split_duplicates = pd.read_csv( os.path.join( eda_output_dir, "cross_split_duplicates.csv"))

print("Training images:", len(train_files))
print("Validation images:", len(val_files))
print("EDA rows:", len(eda_df))
print("Cross-split duplicate rows:", len(cross_split_duplicates))
print("Preprocessing output folder ready:", os.path.exists(preprocessing_output_dir))

Training images: 6600
Validation images: 1800
EDA rows: 8400
Cross-split duplicate rows: 76
Preprocessing output folder ready: True


In [3]:
## 3. Add Numeric Label

metadata_df = eda_df.copy()

metadata_df["label"] = metadata_df["binary_label"].map({"Non-TB": 0, "TB": 1})

print("Metadata rows:", len(metadata_df))
print("Missing labels:", metadata_df["label"].isna().sum())

display(metadata_df[["path","split","original_class","binary_label","label","pixel_hash"]].head())

Metadata rows: 8400
Missing labels: 0


,path,split,original_class,binary_label,label,pixel_hash
0,tb/tb0005.png,train,tb,TB,1,206f531c511e2d861735005e6aa69706099aa929f57be2...
1,tb/tb0007.png,train,tb,TB,1,e157cc364238ece15c6539f692d62d062794608c09b1b5...
2,tb/tb0012.png,train,tb,TB,1,4458c6997d15f93e12f36403b83eb79ffd8930e0eafdd5...
3,tb/tb0017.png,train,tb,TB,1,79d624467d73aec9fbb7b7cdf975c3c7787f543b53a63e...
4,tb/tb0018.png,train,tb,TB,1,554a53780833366816f70cd557c138829a88ccf2e5ece5...


In [4]:
## 4. Remove Cross-Split Training Duplicates

# Separate training and validation duplicate records
train_duplicates = cross_split_duplicates[cross_split_duplicates["split"] == "train"].copy()

validation_duplicates = cross_split_duplicates[cross_split_duplicates["split"] == "validation"][["pixel_hash", "path"]].rename(
    columns={"path": "retained_path"})

# Record removed and retained paths
duplicate_audit = train_duplicates.merge(validation_duplicates,on="pixel_hash")

duplicate_audit["removed_path"] = duplicate_audit["path"]
duplicate_audit["group_size"] = 2
duplicate_audit["class"] = duplicate_audit["binary_label"]
duplicate_audit["removal_reason"] = "Cross-split duplicate"

duplicate_audit = duplicate_audit[[
        "pixel_hash",
        "group_size",
        "retained_path",
        "removed_path",
        "class",
        "removal_reason"
    ]]

# Remove only the training copies
working_df = metadata_df[~metadata_df["path"].isin(train_duplicates["path"])].copy()

print("Training copies removed:", len(train_duplicates))
print("Remaining training images:",(working_df["split"] == "train").sum())
print("Remaining validation images:",(working_df["split"] == "validation").sum())

Training copies removed: 38
Remaining training images: 6562
Remaining validation images: 1800


In [5]:
## 5. Inspect Within-Split Duplicate Groups

# Find duplicate hashes within each split
within_split_duplicates = working_df[working_df.duplicated(subset=["split", "pixel_hash"],keep=False)].copy()

# Summarise each duplicate group
duplicate_groups = within_split_duplicates.groupby(["split", "pixel_hash"]).agg(
    group_size=("path", "size"),
    original_class_count=("original_class", "nunique"),
    binary_label_count=("binary_label", "nunique"),
    binary_label=("binary_label", "first")
).reset_index()

# Identify possible label issues
hard_conflicts = duplicate_groups[duplicate_groups["binary_label_count"] > 1]

soft_mismatches = duplicate_groups[(duplicate_groups["binary_label_count"] == 1) &(duplicate_groups["original_class_count"] > 1)]

print("Within-split duplicate groups:", len(duplicate_groups))
print("Hard-conflict groups:", len(hard_conflicts))
print("Soft-mismatch groups:", len(soft_mismatches))

display(pd.crosstab(duplicate_groups["split"],duplicate_groups["binary_label"]))

Within-split duplicate groups: 88
Hard-conflict groups: 0
Soft-mismatch groups: 0


binary_label,Non-TB
split,
train,82
validation,6


In [6]:
## 6. Remove Within-Split Exact Duplicates

# Sort paths so the alphabetically first copy is retained
sorted_df = working_df.sort_values(["split", "pixel_hash", "path"]).copy()

# Retained image for each duplicate group
retained = sorted_df.drop_duplicates(["split", "pixel_hash"])[["split", "pixel_hash", "path"]]

retained = retained.rename(columns={"path": "retained_path"})

# Identify the copies to remove
within_removed = sorted_df[sorted_df.duplicated(["split", "pixel_hash"],keep="first")].copy()

within_removed = within_removed.merge(retained,on=["split", "pixel_hash"])

# Add these removals to the duplicate audit
within_removed["group_size"] = within_removed.groupby(["split", "pixel_hash"])["path"].transform("size") + 1

within_removed["removed_path"] = within_removed["path"]
within_removed["class"] = within_removed["binary_label"]
within_removed["removal_reason"] = "Within-split exact duplicate"

within_audit = within_removed[[
        "pixel_hash",
        "group_size",
        "retained_path",
        "removed_path",
        "class",
        "removal_reason"
    ]]

duplicate_audit = pd.concat([duplicate_audit, within_audit],ignore_index=True)

# Keep one image per hash within each split
deduplicated_df = sorted_df.drop_duplicates(["split", "pixel_hash"],keep="first").copy()

print("Within-split copies removed:", len(within_removed))
print("Remaining training images:",(deduplicated_df["split"] == "train").sum())
print("Remaining validation images:",(deduplicated_df["split"] == "validation").sum())

Within-split copies removed: 88
Remaining training images: 6480
Remaining validation images: 1794


In [7]:
## 7. Build Final Train and Validation Manifests

final_train_df = deduplicated_df[deduplicated_df["split"] == "train"].copy().reset_index(drop=True)

final_validation_df = deduplicated_df[deduplicated_df["split"] == "validation"].copy().reset_index(drop=True)

print("Final training images:", len(final_train_df))
print("Final validation images:", len(final_validation_df))

Final training images: 6480
Final validation images: 1794


In [8]:
## 8. Validate Final Manifests

# Check for duplicate paths
duplicate_paths = (final_train_df["path"].duplicated().sum() + final_validation_df["path"].duplicated().sum())

# Check for duplicate hashes within each split
duplicate_hashes = (final_train_df["pixel_hash"].duplicated().sum() + final_validation_df["pixel_hash"].duplicated().sum())

# Check for overlap between training and validation
hash_overlap = len(set(final_train_df["pixel_hash"]) & set(final_validation_df["pixel_hash"]))

# Check labels and missing values
invalid_labels = (~pd.concat([final_train_df["label"],final_validation_df["label"]]).isin([0, 1])).sum()

missing_values = (final_train_df.isna().sum().sum() + final_validation_df.isna().sum().sum())

print("Duplicate paths:", duplicate_paths)
print("Within-split duplicate hashes:", duplicate_hashes)
print("Train-validation hash overlap:", hash_overlap)
print("Invalid labels:", invalid_labels)
print("Missing values:", missing_values)

Duplicate paths: 0
Within-split duplicate hashes: 0
Train-validation hash overlap: 0
Invalid labels: 0
Missing values: 0


In [9]:
## 9. Calculate Final Class Counts and Class Weights

from sklearn.utils.class_weight import compute_class_weight

# Final class counts
original_counts = pd.crosstab(deduplicated_df["split"],deduplicated_df["original_class"])

binary_counts = pd.crosstab(deduplicated_df["split"],deduplicated_df["binary_label"])

display(original_counts)
display(binary_counts)

# Class weights from final training data
classes = np.array([0, 1])

weights = compute_class_weight(class_weight="balanced",classes=classes,y=final_train_df["label"])

class_weights = {int(label): float(weight) for label, weight in zip(classes, weights)}

print("Class weights:", class_weights)

original_class,health,sick,tb
split,,,
train,3000,2880,600
validation,800,794,200


binary_label,Non-TB,TB
split,,
train,5880,600
validation,1594,200


Class weights: {0: 0.5510204081632653, 1: 5.4}


In [10]:
## 10. Build Artefact-Audit Manifest

# Combine retained training and validation images
retained_df = pd.concat([final_train_df, final_validation_df], ignore_index=True)

# Set reproducible thresholds
brightness_low = retained_df["mean_intensity"].quantile(0.01)
brightness_high = retained_df["mean_intensity"].quantile(0.99)

contrast_low = retained_df["std_intensity"].quantile(0.01)
contrast_high = retained_df["std_intensity"].quantile(0.99)

border_threshold = retained_df["border_center_diff"].quantile(0.99)
corner_threshold = retained_df["corner_variation"].quantile(0.99)

# Flag unusual images
retained_df["unusual_brightness"] = (
    (retained_df["mean_intensity"] < brightness_low) |
    (retained_df["mean_intensity"] > brightness_high)
)

retained_df["unusual_contrast"] = (
    (retained_df["std_intensity"] < contrast_low) |
    (retained_df["std_intensity"] > contrast_high)
)

retained_df["high_border_difference"] = (
    retained_df["border_center_diff"] > border_threshold
)

retained_df["high_corner_variation"] = (
    retained_df["corner_variation"] > corner_threshold
)

# Keep images requiring later visual review
audit_flags = [
    "unusual_brightness",
    "unusual_contrast",
    "high_border_difference",
    "high_corner_variation"
]

artifact_audit_manifest = retained_df[retained_df[audit_flags].any(axis=1)].copy()

print("Images flagged for artefact review:", len(artifact_audit_manifest))
print("Brightness thresholds:", round(brightness_low, 2), "to", round(brightness_high, 2))
print("Contrast thresholds:", round(contrast_low, 2), "to", round(contrast_high, 2))
print("Border threshold:", round(border_threshold, 2))
print("Corner threshold:", round(corner_threshold, 2))

Images flagged for artefact review: 462
Brightness thresholds: 78.6 to 176.01
Contrast thresholds: 46.1 to 79.3
Border threshold: 98.73
Corner threshold: 89.73


In [11]:
## 11. Save Files Needed Later

final_train_df.to_csv(
    os.path.join(preprocessing_output_dir, "final_train_metadata.csv"),
    index=False
)

final_validation_df.to_csv(
    os.path.join(preprocessing_output_dir, "final_validation_metadata.csv"),
    index=False
)

artifact_audit_manifest.to_csv(
    os.path.join(preprocessing_output_dir, "artifact_audit_manifest.csv"),
    index=False
)

print("Files saved:")
print("final_train_metadata.csv")
print("final_validation_metadata.csv")
print("artifact_audit_manifest.csv")

Files saved:
final_train_metadata.csv
final_validation_metadata.csv
artifact_audit_manifest.csv
